# Thí nghiệm 03 — Financial: Định nghĩa mô hình

## 1. Mục tiêu

Notebook định nghĩa các kiến trúc và hàm dùng chung cho thí nghiệm dự báo chuỗi thời gian Financial.

Thực nghiệm sử dụng hai kênh dữ liệu:

- `BTC_segment_1`
- `BTC_segment_2`

Hai kênh tương ứng với hai đoạn thời gian liên tiếp của Bitcoin, không phải hai đồng tiền BTC và ETH.

### Các mô hình

1. LSTM — mô hình baseline.
2. DeepKoopFormer-PatchTST.
3. DeepKoopFormer-Autoformer.
4. DeepKoopFormer-Informer.

### Các thành phần dùng chung

- Toán tử Koopman ổn định.
- Positional encoding.
- Patch embedding.
- Transformer encoder.
- Series decomposition.
- Hàm tạo sliding windows.
- Hàm mất mát kết hợp chính quy hóa Lyapunov.
- Hàm huấn luyện và đánh giá.

## 1.1. Phạm vi notebook

Notebook chỉ định nghĩa kiến trúc và các hàm cần thiết.

Dữ liệu đầu vào được chuẩn bị trong `01_financial_data.ipynb`. Quy trình Grid Search và quản lý checkpoint nằm trong `03_financial_training.ipynb`.

Không thực hiện huấn luyện hoặc kiểm tra ngẫu nhiên trong notebook này.

## 1.2. Khả năng tương thích

Các định nghĩa phải giữ nguyên giao diện và phép tính so với phiên bản đã sử dụng để huấn luyện, nhằm duy trì khả năng nạp các checkpoint hiện có.


In [1]:
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim


# 1. Thiết lập seed
SEED = 7

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# 2. Thiết bị tính toán
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 3. Thông báo cấu hình
print("=== FINANCIAL MODEL DEFINITIONS ===")
print("Seed:", SEED)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

=== FINANCIAL MODEL DEFINITIONS ===
Seed: 7
Device: cuda
PyTorch: 2.0.1+cu118



## 2. Toán tử Koopman ổn định

### 2.1. Vai trò

Toán tử Koopman được sử dụng để biến đổi biểu diễn trong không gian tiềm ẩn.

Với biểu diễn \(z\), phép biến đổi có dạng:

\[
z_{\mathrm{next}} = zK^T
\]

Trong triển khai hiện tại, toán tử \(K\) được tham số hóa bởi:

\[
K = U\Sigma V^T
\]

Trong đó:

- \(U\) và \(V\) được trực giao hóa bằng QR decomposition.
- \(\Sigma\) là ma trận đường chéo.
- Các giá trị trên đường chéo được giới hạn thông qua sigmoid.

Cụ thể:

\[
\sigma_i = \rho_{\max}\operatorname{sigmoid}(s_i)
\]

với:

\[
\rho_{\max}=0.99
\]

Do đó, các giá trị kỳ dị của \(K\) được ràng buộc nhỏ hơn 0,99 trong điều kiện số học lý tưởng, giúp giới hạn độ khuếch đại của phép biến đổi tuyến tính.

### 2.2. Các thành phần

- `orthogonalize()`: tạo ma trận trực giao bằng QR.
- `StrictStableKoopmanOperator`: xây dựng toán tử và biến đổi latent representation.

### 2.3. Đầu ra

Hàm `forward()` trả về:

- `z_next`: biểu diễn sau phép biến đổi.
- `K`: toán tử Koopman.
- `Sigma`: vector giá trị kỳ dị bị ràng buộc.

Toán tử này được sử dụng trong ba biến thể DeepKoopFormer, không sử dụng trong baseline LSTM.

In [3]:
def orthogonalize(W: torch.Tensor) -> torch.Tensor:
    Q, R = torch.linalg.qr(W)
    return Q


class StrictStableKoopmanOperator(nn.Module):

    def __init__(self, latent_dim: int, rho_max: float = 0.99):
        super().__init__()

        self.U_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.V_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.S_raw = nn.Parameter(
            torch.randn(latent_dim)
        )

        self.rho_max = rho_max

    def forward(self, z: torch.Tensor):

        # Orthogonal matrices
        U = orthogonalize(self.U_raw)
        V = orthogonalize(self.V_raw)

        # Constrained singular values
        Sigma = torch.sigmoid(self.S_raw) * self.rho_max

        # Construct Koopman operator
        K = U @ torch.diag(Sigma) @ V.T

        # Transform latent representations
        z_next = z @ K.T

        return z_next, K, Sigma


## 3. Sinusoidal Positional Encoding

### 3.1. Mục đích

Transformer không tự động biểu diễn vị trí của từng token trong chuỗi.

`SinCosPosEnc` bổ sung thông tin vị trí bằng hai hàm sin và cos ở các tần số khác nhau.

Với vị trí \(pos\) và chiều đặc trưng \(i\), positional encoding có dạng:

\[
PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d}}\right)
\]

\[
PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d}}\right)
\]

### 3.2. Triển khai

- Tạo positional encoding khi khởi tạo module.
- Lưu bằng `register_buffer()` thay vì tham số học được.
- Cộng positional encoding vào embedding đầu vào trong `forward()`.

### 3.3. Sử dụng

Thành phần này được dùng trong các backbone PatchTST, Autoformer và Informer của thí nghiệm.


In [5]:
class SinCosPosEnc(nn.Module):

    def __init__(self, d_model: int, max_len: int = 10000):
        super().__init__()

        # Vị trí từ 0 đến max_len - 1
        pos = torch.arange(max_len).float().unsqueeze(1)

        # Hệ số tần số cho từng cặp chiều
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        # Khởi tạo positional encoding
        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        # Lưu vào model nhưng không coi là trainable parameter
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]


## 4. Patch Embedding

### 4.1. Mục đích

`PatchEmbed1D` chuyển chuỗi thời gian đa biến thành chuỗi embedding có kích thước `d_model`.

Đầu vào:

\[
X\in\mathbb{R}^{B\times T\times C}
\]

Trước khi áp dụng `Conv1d`, tensor được chuyển thành:

\[
X'\in\mathbb{R}^{B\times C\times T}
\]

Sau phép tích chập, đầu ra được chuyển về:

\[
E\in\mathbb{R}^{B\times N_p\times d_{\mathrm{model}}}
\]

Trong đó \(N_p\) là số patch.

### 4.2. Cấu hình

Patch embedding sử dụng `nn.Conv1d` với kernel size và stride được truyền từ backbone.

Trong `PatchTST_Backbone`, stride được đặt bằng patch length, tức là các patch không chồng lấn.

Cấu hình thực nghiệm sử dụng `seq_len = patch_len`, vì vậy mỗi đầu vào tạo ra một patch trước khi thêm CLS token.

### 4.3. Lưu ý

Không thay `stride`, `kernel_size` hoặc cách hoán vị tensor vì những thay đổi này ảnh hưởng đến đầu ra của mô hình.


In [6]:
class PatchEmbed1D(nn.Module):

    def __init__(
        self,
        in_ch: int,
        d_model: int,
        patch_len: int,
        stride: int
    ):
        super().__init__()

        self.conv = nn.Conv1d(
            in_channels=in_ch,
            out_channels=d_model,
            kernel_size=patch_len,
            stride=stride
        )

    def forward(self, x):

        # (Batch, Time, Channels) -> (Batch, Channels, Time)
        x = x.permute(0, 2, 1)

        # Convolution tạo patch embeddings
        x = self.conv(x)

        # (Batch, d_model, N_patches) -> (Batch, N_patches, d_model)
        x = x.permute(0, 2, 1)

        return x


## 5. PatchTST Backbone

### 5.1. Kiến trúc

`PatchTST_Backbone` gồm:

1. Patch embedding.
2. Sinusoidal positional encoding.
3. Transformer encoder.
4. CLS token học được.

Sau Transformer encoder, biểu diễn ở vị trí CLS token được lấy làm latent representation.

### 5.2. Cấu hình mặc định

| Tham số | Giá trị |
|---|---|
| `d_model` | 48 |
| `num_layers` | 3 |
| `num_heads` | 4 |
| `dim_ff` | 96 |
| Patch stride | Bằng patch length |

### 5.3. Đầu ra

Backbone trả về một vector latent có kích thước `d_model` cho mỗi mẫu.

Vector này được chuyển sang toán tử Koopman trong mô hình DeepKoopFormer-PatchTST.

### 5.4. Phạm vi triển khai

Backbone ở đây là phiên bản được hiện thực hóa cho thí nghiệm tái lập; không mặc định đồng nhất mọi thành phần với thư viện PatchTST gốc.


In [8]:
class PatchTST_Backbone(nn.Module):

    def __init__(
        self,
        input_dim: int,
        seq_len: int,
        patch_len: int,
        d_model: int = 48,
        num_layers: int = 3,
        num_heads: int = 4,
        dim_ff: int = 96
    ):
        super().__init__()

        # 1. Chia chuỗi thành các patch và embedding
        self.patch = PatchEmbed1D(
            in_ch=input_dim,
            d_model=d_model,
            patch_len=patch_len,
            stride=patch_len
        )

        # 2. Positional Encoding
        n_patches = int(np.ceil(seq_len / patch_len))

        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=n_patches + 1
        )

        # 3. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 4. Learnable CLS Token
        self.cls = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

    def forward(self, x):

        # (B, seq_len, input_dim) -> (B, n_patches, d_model)
        x = self.patch(x)

        # Mở rộng CLS token cho toàn bộ batch
        cls = self.cls.expand(x.size(0), -1, -1)

        # Ghép CLS token vào đầu chuỗi
        x = torch.cat([cls, x], dim=1)

        # Positional Encoding + Transformer Encoder
        x = self.encoder(self.pos(x))

        # Lấy biểu diễn tại CLS token
        z = x[:, 0]

        return z


## 6. DeepKoopFormer-PatchTST

### 6.1. Quy trình dự báo

Mô hình gồm ba giai đoạn:

1. `PatchTST_Backbone` mã hóa chuỗi đầu vào thành latent representation \(z\).
2. `StrictStableKoopmanOperator` biến đổi \(z\) thành \(z_{\mathrm{next}}\).
3. Linear forecasting head tạo dự báo cho hai kênh BTC.

Đầu ra:

\[
\hat{Y}\in\mathbb{R}^{B\times(2H)}
\]

### 6.2. Chế độ trả về latent

Khi `return_latents=False`, mô hình chỉ trả về dự báo.

Khi `return_latents=True`, mô hình trả về:

- `pred`
- `z`
- `z_next`

Hai latent representation được sử dụng trong hàm mất mát chính quy hóa Lyapunov.

### 6.3. Lưu ý tương thích

Giữ nguyên tên thuộc tính `backbone`, `koop` và `fc` để tương thích với `model_state_dict` trong các checkpoint.


In [9]:
class Koopformer_PatchTST(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=48
    ):
        super().__init__()

        # PatchTST backbone
        self.backbone = PatchTST_Backbone(
            input_dim=input_dim,
            seq_len=seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        # Koopman operator
        self.koop = StrictStableKoopmanOperator(
            latent_dim=d_model
        )

        # Forecasting head
        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # Encode input into latent space
        z = self.backbone(x)

        # Apply Koopman operator
        z_next, K, Sigma = self.koop(z)

        # Forecast future values
        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred


## 7. Phân rã chuỗi thời gian — Series Decomposition

### 7.1. Mục đích

Chuỗi thời gian đầu vào được phân tách thành hai thành phần:

- Trend: xu hướng được ước lượng bằng moving average.
- Seasonal: phần còn lại sau khi trừ trend.

\[
X_{\mathrm{seasonal}} = X-X_{\mathrm{trend}}
\]

### 7.2. Triển khai

Lớp `SeriesDecomp` sử dụng `nn.AvgPool1d` để tính trung bình trượt.

Đầu vào và đầu ra đều có dạng:

\[
(B,T,C)
\]

Trong `SimpleAutoformer`, độ dài kernel của moving average được đặt bằng `patch_len`.

### 7.3. Lưu ý

Với kernel chẵn, phép pooling có thể sinh ra nhiều hơn một thời điểm. Code hiện tại cắt phần dư để giữ kích thước thời gian bằng đầu vào.

Không thay cách padding hoặc cắt chiều thời gian vì điều này ảnh hưởng đến biểu diễn mà mô hình đã học.


In [12]:
class SeriesDecomp(nn.Module):

    def __init__(self, k: int = 3):
        super().__init__()

        self.avg = nn.AvgPool1d(
            kernel_size=k,
            stride=1,
            padding=k // 2
        )

    def forward(self, x):

        # (B, T, C) -> (B, C, T)
        x_transposed = x.transpose(1, 2)

        # Moving Average -> Trend
        trend = self.avg(x_transposed)

        # (B, C, T) -> (B, T, C)
        trend = trend.transpose(1, 2)

        # Với kernel chẵn, AvgPool có thể sinh thêm 1 time step
        if trend.size(1) != x.size(1):
            trend = trend[:, :x.size(1)]

        # Seasonal = Original - Trend
        seasonal = x - trend

        return trend, seasonal


## 8. Simplified Autoformer Backbone

### 8.1. Kiến trúc

`SimpleAutoformer` xử lý hai thành phần của chuỗi:

**Seasonal branch**

- Chiếu đặc trưng đầu vào sang `d_model`.
- Thêm positional encoding.
- Đưa qua Transformer encoder.
- Mean pooling theo chiều thời gian.
- Linear head dự báo `horizon` giá trị.

**Trend branch**

- Làm phẳng tensor trend.
- Dùng linear layer để dự báo `horizon` giá trị.

Đầu ra cuối cùng được tính bằng tổng hai nhánh:

\[
z = \operatorname{SeasonalBranch}(X)
+\operatorname{TrendBranch}(X)
\]

Với:

\[
z\in\mathbb{R}^{B\times H}
\]

### 8.2. Cấu hình

- `d_model = 48`.
- `num_heads = 4`.
- `num_layers = 3`.
- `dim_ff = 64`.

### 8.3. Phạm vi triển khai

Đây là một backbone Autoformer đơn giản hóa, sử dụng decomposition và Transformer encoder thông thường.

Code không triển khai đầy đủ cơ chế Auto-Correlation của Autoformer gốc.

Do đó, trong báo cáo cần mô tả đúng kiến trúc đã hiện thực hóa thay vì khẳng định đây là nguyên bản Autoformer.


In [13]:
class SimpleAutoformer(nn.Module):

    def __init__(
        self,
        input_len,
        horizon,
        input_dim,
        patch_len,
        d_model=48,
        num_heads=4,
        dim_ff=64,
        num_layers=3
    ):
        super().__init__()

        # 1. Series Decomposition
        self.dec = SeriesDecomp(k=patch_len)

        # 2. Seasonal branch
        self.embed = nn.Linear(input_dim, d_model)

        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=input_len
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 3. Forecasting heads for both components
        self.fc_seas = nn.Linear(d_model, horizon)

        self.fc_trend = nn.Linear(
            input_len * input_dim,
            horizon
        )

    def forward(self, x):

        # Separate trend and seasonal components
        trend, seas = self.dec(x)

        # Seasonal branch: Transformer Encoder
        seas = self.embed(seas)
        seas = self.pos(seas)
        seas = self.encoder(seas)

        # Aggregate seasonal representations
        seas_o = self.fc_seas(seas.mean(dim=1))

        # Trend branch: Flatten + Linear
        trend_flat = trend.reshape(trend.size(0), -1)
        trend_o = self.fc_trend(trend_flat)

        # Combine both branches
        return seas_o + trend_o


## 9. DeepKoopFormer-Autoformer

### 9.1. Quy trình

1. `SimpleAutoformer` tạo latent representation.
2. Toán tử Koopman biến đổi latent representation.
3. Linear head tạo dự báo đa biến.

### 9.2. Kích thước latent

Khác với PatchTST và Informer, backbone Autoformer hiện tại trả về vector có kích thước bằng forecast horizon:

\[
z\in\mathbb{R}^{B\times H}
\]

Do đó:

\[
K\in\mathbb{R}^{H\times H}
\]

Forecasting head biến đổi từ \(H\) sang \(2H\), ứng với hai kênh Financial.

### 9.3. Tương thích checkpoint

Không thay latent dimension từ `horizon` thành `d_model`.

Đây là một khác biệt kiến trúc quan trọng; sửa nó sẽ làm thay đổi kích thước tham số của mô hình và không còn tương thích với checkpoint hiện tại.


In [14]:
class Koopformer_Autoformer(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=48
    ):
        super().__init__()

        # Autoformer backbone
        self.backbone = SimpleAutoformer(
            input_len=seq_len,
            horizon=horizon,
            input_dim=input_dim,
            patch_len=patch_len,
            d_model=d_model
        )

        # Koopman operates in H-dimensional latent space
        self.koop = StrictStableKoopmanOperator(
            latent_dim=horizon
        )

        # Forecasting head
        self.fc = nn.Linear(
            horizon,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)

        z_next, K, Sigma = self.koop(z)

        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred


## 10. Informer Backbone

### 10.1. Vai trò

Lớp `InformerSparse` tạo latent representation từ chuỗi thời gian đầu vào.

Quy trình:

1. Input embedding.
2. Sinusoidal positional encoding.
3. Transformer encoder.
4. Mean pooling theo chiều token.

### 10.2. Input embedding

Nếu `patch_len > 1`, mô hình sử dụng `PatchEmbed1D`.

Nếu `patch_len = 1`, mô hình sử dụng một Linear embedding cho từng thời điểm.

### 10.3. Transformer Encoder

Cấu hình mặc định:

- `d_model = 48`.
- `num_heads = 4`.
- `num_layers = 3`.
- `dim_ff = 96`.

Đầu ra sau mean pooling có kích thước `d_model`.

### 10.4. Phạm vi triển khai

Mặc dù lớp được đặt tên là `InformerSparse`, code hiện tại sử dụng `nn.TransformerEncoderLayer`.

Nó không triển khai cơ chế ProbSparse attention đặc trưng của Informer gốc.

Điều này cần được ghi nhận khi mô tả mô hình trong báo cáo tái lập.


In [16]:
class InformerSparse(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=48,
        num_heads=4,
        dim_ff=96,
        num_layers=3,
        seq_len=120,
        patch_len=1
    ):
        super().__init__()

        # Có sử dụng Patch Embedding hay không?
        self.use_patch = patch_len > 1

        if self.use_patch:
            self.patch = PatchEmbed1D(
                in_ch=input_dim,
                d_model=d_model,
                patch_len=patch_len,
                stride=patch_len
            )

            n_tokens = int(np.ceil(seq_len / patch_len))

        else:
            self.embed = nn.Linear(
                input_dim,
                d_model
            )

            n_tokens = seq_len

        # Positional Encoding
        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=n_tokens
        )

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        # 1. Input Embedding
        if self.use_patch:
            x = self.patch(x)
        else:
            x = self.embed(x)

        # 2. Positional Encoding
        x = self.pos(x)

        # 3. Transformer Encoder
        x = self.encoder(x)

        # 4. Mean Pooling across tokens
        z = x.mean(dim=1)

        return z


## 11. DeepKoopFormer-Informer

### 11.1. Kiến trúc

Mô hình kết hợp:

1. `InformerSparse` để mã hóa chuỗi thời gian.
2. `StrictStableKoopmanOperator` để biến đổi latent representation.
3. Linear forecasting head để dự báo hai segment BTC.

### 11.2. Latent representation

Trong triển khai này:

\[
z\in\mathbb{R}^{B\times d_{\mathrm{model}}}
\]

Toán tử Koopman có kích thước:

\[
K\in\mathbb{R}^{d_{\mathrm{model}}\times d_{\mathrm{model}}}
\]

Forecasting head trả về:

\[
\hat Y\in\mathbb{R}^{B\times(2H)}
\]

### 11.3. Chế độ trả về latent

Khi `return_latents=True`, mô hình trả về `pred`, `z`, `z_next` để phục vụ tính loss Lyapunov.

Giữ nguyên thuộc tính mô hình và cấu trúc đầu ra để bảo toàn tương thích checkpoint.


In [17]:
class Koopformer_Informer(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=48
    ):
        super().__init__()

        # Informer Backbone
        self.backbone = InformerSparse(
            input_dim=input_dim,
            d_model=d_model,
            seq_len=seq_len,
            patch_len=patch_len
        )

        # Koopman Operator
        self.koop = StrictStableKoopmanOperator(
            latent_dim=d_model
        )

        # Forecasting Head
        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # Extract latent representation
        z = self.backbone(x)

        # Apply Koopman transformation
        z_next, K, Sigma = self.koop(z)

        # Forecast BTC and ETH
        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred


## 12. Baseline LSTM

### 12.1. Mục đích

LSTM được sử dụng làm mô hình baseline để so sánh với ba biến thể DeepKoopFormer.

### 12.2. Kiến trúc

`SimpleLSTMForecaster` gồm:

- Các lớp LSTM xử lý chuỗi đầu vào.
- Hidden state cuối cùng của lớp LSTM cuối.
- Linear forecasting head.

Đầu ra:

\[
\hat Y\in\mathbb{R}^{B\times(2H)}
\]

Trong Grid Search Financial, mô hình được khởi tạo với hai lớp LSTM và hidden dimension bằng 48.

### 12.3. Khác biệt so với DeepKoopFormer

LSTM không chứa toán tử Koopman.

Khi `return_latents=True`, mô hình trả về:

`pred, z, z`

Tức `z_next` trùng với `z`. Vì vậy, thành phần phạt Lyapunov dựa trên hiệu năng lượng giữa hai biểu diễn này bằng 0.

Đây là một quy ước trong triển khai hiện tại, không phải phép biến đổi Koopman thực sự của LSTM.


In [19]:
class SimpleLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers,
        horizon
    ):
        super().__init__()

        # 1. LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # 2. Forecasting head
        self.fc = nn.Linear(
            hidden_dim,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # h_n: (num_layers, batch_size, hidden_dim)
        _, (h_n, c_n) = self.lstm(x)

        # Final hidden state of the last LSTM layer
        z = h_n[-1]

        # Forecast future values
        pred = self.fc(z)

        if return_latents:
            return pred, z, z

        return pred


## 13. Hàm tạo sliding windows

### 13.1. Mục đích

Hàm `create_windows()` chuyển chuỗi thời gian hai kênh thành các cặp input-target.

Với input length \(P\) và forecast horizon \(H\):

\[
X_i\in\mathbb{R}^{P\times2}
\]

\[
y_i\in\mathbb{R}^{2H}
\]

### 13.2. Quy tắc làm phẳng nhãn

Nhãn được tạo bằng:

`target.T.reshape(-1)`

Thứ tự của nhãn là channel-major:

1. Toàn bộ \(H\) giá trị tương lai của segment 1.
2. Toàn bộ \(H\) giá trị tương lai của segment 2.

### 13.3. Số lượng cửa sổ

Hàm sử dụng đúng:

`range(len(data) - seq_len - horizon)`

Không thêm `+1` vì cần giữ nguyên số lượng cửa sổ đã dùng trong thực nghiệm.

### 13.4. Đầu vào

Hàm này nhận chuỗi thời gian ở dạng PyTorch tensor. Tập Train và Test được nạp từ bundle dữ liệu ở notebook Training.

Trong notebook Model, chỉ định nghĩa hàm, không tạo hoặc lưu các cửa sổ thử nghiệm.


In [2]:
def create_windows(data, seq_len, horizon):
    X, y = [], []

    for i in range(len(data) - seq_len - horizon):

        # p historical timestamps
        x_window = data[i:i + seq_len]

        # H future timestamps for each channel
        y_window = data[
            i + seq_len:i + seq_len + horizon
        ].T.reshape(-1)

        X.append(x_window)
        y.append(y_window)

    X = torch.stack(X).float()
    y = torch.stack(y).float()

    return X, y


## 14. Hàm mất mát và chính quy hóa Lyapunov

### 14.1. Prediction Loss

Sai số dự báo được đo bằng MSE:

\[
\mathcal{L}_{pred}
=
\frac{1}{N}\sum_{i=1}^{N}
\left(\hat y_i-y_i\right)^2
\]

### 14.2. Lyapunov Regularization

Trong triển khai hiện tại, sử dụng hàm năng lượng:

\[
V(z)=z^TPz
\]

với:

\[
P=I
\]

Thành phần chính quy hóa phạt sự gia tăng năng lượng sau biến đổi Koopman:

\[
\mathcal{L}_{Lyap}
=
\frac{1}{B}\sum_{i=1}^{B}
\max\left(0,V(z_{\mathrm{next},i})-V(z_i)\right)
\]

### 14.3. Total Loss

\[
\mathcal{L}
=
\mathcal{L}_{pred}
+
\lambda_{Lyap}\mathcal{L}_{Lyap}
\]

Mặc định:

\[
\lambda_{Lyap}=0.1
\]

Khi hệ số này bằng 0, hàm mất mát chỉ còn MSE.

### 14.4. Đầu ra

`compute_loss()` trả về:

- `total_loss`
- `mse_loss`
- `lyap_loss`

Giữ nguyên công thức, cách tính trung bình và kiểu tensor để bảo đảm tương thích với quá trình huấn luyện đã thực hiện.


In [22]:
def compute_loss(
    pred,
    target,
    z,
    z_next,
    lyap_weight=0.1
):
    # 1. Prediction loss
    mse_loss = nn.functional.mse_loss(pred, target)

    # 2. Lyapunov regularization
    if lyap_weight > 0:

        # P = Identity matrix
        P = torch.eye(
            z.shape[1],
            device=z.device,
            dtype=z.dtype
        )

        # V(z) = z^T P z
        V_z = torch.einsum("bi,ij,bj->b", z, P, z)

        # V(z_next) = z_next^T P z_next
        V_z_next = torch.einsum(
            "bi,ij,bj->b",
            z_next, P, z_next
        )

        # Penalize positive increase
        lyap_loss = torch.relu(
            V_z_next - V_z
        ).mean()

    else:
        lyap_loss = torch.zeros_like(mse_loss)

    # 3. Total loss
    total_loss = mse_loss + lyap_weight * lyap_loss

    return total_loss, mse_loss, lyap_loss


## 15. Hàm huấn luyện mô hình

### 15.1. Mục đích

`train_model()` thực hiện tối ưu trọng số cho một mô hình và một cấu hình dự báo cụ thể.

Hàm được sử dụng bởi quy trình Grid Search trong `03_financial_training.ipynb`.

### 15.2. Quy trình

1. Chuyển mô hình và dữ liệu lên thiết bị tính toán.
2. Khởi tạo Adam optimizer.
3. Chạy tối ưu trong số epoch được chỉ định.
4. Tính Prediction Loss và Lyapunov Loss.
5. Backpropagation và cập nhật tham số.
6. Ghi nhận lịch sử huấn luyện.

### 15.3. Cấu hình mặc định

| Tham số | Giá trị |
|---|---|
| Epochs | 4000 |
| Learning rate | 0.0003 |
| Lyapunov weight | 0.1 |
| Optimizer | Adam |
| Training mode | Full-batch |

Với baseline LSTM, quy trình Grid Search truyền hệ số Lyapunov bằng 0.

### 15.4. Lịch sử huấn luyện

Hàm trả về mô hình sau tối ưu và bốn chuỗi:

- `loss`: Total Loss.
- `mse`: Prediction Loss.
- `lyap`: Lyapunov Loss.
- `eig`: Giá trị lớn nhất của `sigmoid(S_raw)` đối với mô hình Koopman.

Tên `eig` là tên biến được sử dụng trong checkpoint, nhưng đại lượng đang theo dõi **không phải trị riêng thực tế của ma trận Koopman**.

### 15.5. Lưu ý

Notebook này chỉ định nghĩa hàm `train_model()`, không gọi hàm để thực hiện huấn luyện.

Việc thực thi 224 cấu hình và lưu checkpoint thuộc notebook Training.


In [24]:
def train_model(
    model,
    X_train,
    y_train,
    epochs=4000,
    lr=3e-4,
    lyap_weight=0.1,
    verbose=True
):
    model = model.to(DEVICE)

    X_train = X_train.to(DEVICE)
    y_train = y_train.to(DEVICE)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    # Lưu lịch sử huấn luyện
    loss_history = []
    mse_history = []
    lyap_history = []
    eig_history = []

    log_interval = max(1, epochs // 10)

    for epoch in range(epochs):

        model.train()

        # 1. Reset gradients
        optimizer.zero_grad()

        # 2. Forward pass
        pred, z, z_next = model(
            X_train,
            return_latents=True
        )

        # 3. Compute loss
        total_loss, mse_loss, lyap_loss = compute_loss(
            pred=pred,
            target=y_train,
            z=z,
            z_next=z_next,
            lyap_weight=lyap_weight
        )

        # 4. Backpropagation
        total_loss.backward()

        # 5. Update parameters
        optimizer.step()

        # 6. Save training history
        loss_history.append(total_loss.item())
        mse_history.append(mse_loss.item())
        lyap_history.append(lyap_loss.item())

        # Theo dõi đại lượng giống script tác giả:
        # max(sigmoid(S_raw)), không phải eigenvalue thực của K
        if hasattr(model, "koop"):
            with torch.no_grad():
                eig_max = (
                    torch.sigmoid(model.koop.S_raw)
                    .abs()
                    .max()
                    .item()
                )
        else:
            eig_max = 0.0

        eig_history.append(eig_max)

        if verbose and (
            epoch % log_interval == 0
            or epoch == epochs - 1
        ):
            print(
                f"Epoch {epoch+1:4d}/{epochs} | "
                f"MSE: {mse_loss.item():.6f} | "
                f"Lyap: {lyap_loss.item():.6f} | "
                f"Total: {total_loss.item():.6f}"
            )

    return {
        "model": model,
        "loss": np.array(loss_history),
        "mse": np.array(mse_history),
        "lyap": np.array(lyap_history),
        "eig": np.array(eig_history)
    }


## 16. Hàm đánh giá mô hình

### 16.1. Mục đích

`evaluate_model()` tính MSE, MAE và dự báo của mô hình trên một tập dữ liệu đã chuẩn bị.

Hàm được sử dụng sau khi huấn luyện để đánh giá Train và Test.

### 16.2. Quy trình

1. Chuyển mô hình sang `eval()`.
2. Tắt tính gradient bằng `torch.no_grad()`.
3. Chạy dự báo theo các batch.
4. Tính MSE và MAE.
5. Trả về metric cùng tensor dự báo.

### 16.3. Các metric

\[
MSE=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2
\]

\[
MAE=\frac{1}{N}\sum_{i=1}^{N}|\hat y_i-y_i|
\]

### 16.4. Trọng số mô hình

Hàm đánh giá không thực hiện backward propagation và không gọi optimizer.

Do đó, việc đánh giá không cập nhật trọng số đã học.

### 16.5. Đầu ra

Hàm trả về dictionary gồm:

- `MSE`
- `MAE`
- `predictions`

Notebook Model chỉ định nghĩa hàm. Việc đánh giá checkpoint và lưu các kết quả thuộc notebook Training.


In [3]:
def evaluate_model(model, X, y, batch_size=256):
    model.eval()

    device = next(model.parameters()).device

    all_predictions = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            x_batch = X[i:i + batch_size].to(device)

            pred = model(x_batch)

            all_predictions.append(pred.cpu())

    predictions = torch.cat(all_predictions, dim=0)

    # Calculate evaluation metrics
    mse = torch.mean((predictions - y) ** 2).item()
    mae = torch.mean(torch.abs(predictions - y)).item()

    return {
        "MSE": mse,
        "MAE": mae,
        "predictions": predictions
    }